We have found that the detection of an event using QR decompsition. This notebook is a simple direct illustration of the method.

In [ ]:
import time

import h5py
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def loadBradyHShdf5(file, normalize="yes"):
    """

    Parameters
    ----------
    file : str
        path to brady hotspring h5py data file
    normalize : str, optional
        "yes" or "no". Indicates whether or not to remove laser drift and
        normalize. The default is 'yes'.

    Returns
    -------
    data : np array
        channel by samples numpy array of data
    timestamp_arr : numpy array
        array of the timestamps corresponding to the various samples in the
        data. Timestamps for brady hotspring data are with respect to the
        beginning time of the survey.

    """
    with h5py.File(file, "r") as open_file:
        dataset = open_file["das"]
        time = open_file["t"]
        data = np.array(dataset)
        timestamp_arr = np.array(time)
    data = np.transpose(data)
    if normalize == "yes":
        nSamples = np.shape(data)[1]
        # get rid of laser drift
        med = np.median(data, axis=0)
        for i in range(nSamples):
            data[:, i] = data[:, i] - med[i]

        max_of_rows = abs(data[:, :]).sum(axis=1)
        data = data / max_of_rows[:, np.newaxis]
    return data, timestamp_arr


def windowed_spectra(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    """
    Calculate the frequency domain representation of data in windows.
    """
    win_start = 0
    window_samples = int(subwindow_len / sample_interval)
    total_samples = data.shape[-1]
    overlap = int(overlap / sample_interval)
    intervals = np.arange(
        window_samples, total_samples + 1, window_samples, dtype=int
    )  # break time series into windowed intervals

    win_end = intervals[0]

    absolute_spectra = np.fft.rfft(data[:, win_start:win_end])
    win_spectra = absolute_spectra[np.newaxis]

    while win_end < total_samples:
        win_start = win_end - overlap
        win_end = win_start + window_samples
        absolute_spectra = np.fft.rfft(data[:, win_start:win_end])
        win_spectra = np.append(
            win_spectra, absolute_spectra[np.newaxis], axis=0
        )
        # win_start = win_end

    frequencies = np.fft.rfftfreq(window_samples, sample_interval)

    return win_spectra, frequencies


def normalised_windowed_spectra(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    win_spectra, frequencies = windowed_spectra(
        data, subwindow_len, overlap, freq, sample_interval
    )

    # win_spectra = np.absolute(win_spectra)**2 # sub for next line
    # win_spectra = win_spectra * np.conjugate(win_spectra) # absolutes square of spectra. We need this if
    # we want to use the normalised spectra to calculate welch coherence.
    # normalizer = np.sum(win_spectra, axis=0)

    normalizer = np.sum(np.absolute(win_spectra) ** 2, axis=0)
    normalizer = np.tile(np.sqrt(normalizer), (win_spectra.shape[0], 1, 1))
    normalizer = normalizer.transpose(2, 1, 0)

    normalized_spectra = win_spectra.transpose(2, 1, 0) / normalizer

    return normalized_spectra, frequencies


def welch_coherence(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    """
    Calculate the coherence matrix at all (or particular frequencies: yet to be implemented)
    using the welch method.
    """
    win_spectra, frequencies = windowed_spectra(
        data, subwindow_len, overlap, freq, sample_interval
    )

    normalizer = np.sum(np.absolute(win_spectra) ** 2, axis=0)
    normalizer = np.tile(normalizer, (normalizer.shape[0], 1, 1))
    normalizer = normalizer * normalizer.transpose((1, 0, 2))
    normalizer = normalizer.transpose(2, 1, 0)

    welch_numerator = np.matmul(
        win_spectra.transpose(2, 1, 0),
        np.conjugate(win_spectra.transpose(2, 0, 1)),
    )
    welch_numerator = np.absolute(welch_numerator) ** 2
    coherence = np.multiply(welch_numerator, 1 / normalizer)

    return coherence, frequencies


def frequency_filter(data, frequency_range, mode, order, sampling_frequency):
    """
    Butterworth filter of data.

    Parameters
    ----------
    data : array
        1d or 2d array.
    frequency_range : int/sequence
        int if mode is lowpass or high pass. Sequence of 2 frequencies if mode
        is bandpass
    mode : str
        lowpass, highpass or bandpass.
    order : int
        Order of the filter.
    sampling_frequency : int
        sampling frequency.

    Returns
    -------
    filtered_data : array
        Frequency filtered data.

    """
    from scipy.signal import butter, sosfiltfilt

    sos = butter(
        order, frequency_range, btype=mode, output="sos", fs=sampling_frequency
    )
    filtered_data = sosfiltfilt(sos, data)

    return filtered_data

In [ ]:
file = r"D:\CSM\Mines_Research\Test_data\Brady Hotspring\PoroTomo_iDAS16043_160314083818.h5"
data3, _ = loadBradyHShdf5(file, normalize="no")

file = r"D:\CSM\Mines_Research\Test_data\Brady Hotspring\PoroTomo_iDAS16043_160314083848.h5"
data, _ = loadBradyHShdf5(file, normalize="no")

file = r"D:\CSM\Mines_Research\Test_data\Brady Hotspring\PoroTomo_iDAS16043_160314083918.h5"
data2, _ = loadBradyHShdf5(file, normalize="no")

# signalToUse=np.append(data[:,24976:],data2[:,:10000],axis=1)
data = np.append(data, data2, axis=1)
data = np.append(data, data3, axis=1)
samples_per_sec = 1000

In [ ]:
# clean channels
# start_ch = 1000
# nchannels = 3000
# including channel with noise
# start_ch = 3100
start_ch = 5255
nchannels = 1100
nsensors = 200

In [ ]:
fsize = 15

pdata = data[start_ch : nchannels + start_ch, 25000:]
# pdata = frequency_filter(pdata, [1, 15], "bandpass", 5, samples_per_sec)

# pdata=data_noise[:,:15000] # signalToUse[1900:3900]

fig2 = plt.figure()
img2 = plt.imshow(
    pdata,
    cmap="RdBu",
    vmin=-np.percentile(np.absolute(pdata), 90),
    vmax=np.percentile(np.absolute(pdata), 90),
    aspect="auto",
    interpolation="none",
    extent=(
        0,
        len(pdata[0]) / samples_per_sec,
        start_ch,
        start_ch + nchannels,
    ),
)
# extent=(mdates.date2num(np.datetime64(props['GPSTimeStamp'])),mdates.date2num(np.datetime64(props['GPSTimeStamp'])+np.timedelta64(60,'s')), distances[0],distances[-1]))
plt.xlabel("Time (seconds)", fontsize=fsize)
plt.ylabel("Sensors", fontsize=fsize)
plt.title("Signal", fontsize=fsize)
plt.xticks(fontsize=fsize)
plt.yticks(fontsize=fsize)
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=fsize)

In [ ]:
norm_win_spectra, frequencies = normalised_windowed_spectra(
    data[start_ch : nchannels + start_ch : int(nchannels / nsensors), 25000:],
    5,
    2.5,
    sample_interval=0.001,
)
# norm_win_spectra, frequencies = normalised_windowed_spectra(data[start_ch:nchannels+start_ch:int(nchannels/nsensors)], 5, 2.5, sample_interval=0.001)

# welch_coherence = np.matmul(norm_win_spectra.transpose(2,1,0), np.conjugate(norm_win_spectra.transpose(2,0,1)))
welch_coherence_mat = np.matmul(
    norm_win_spectra, np.conjugate(norm_win_spectra.transpose(0, 2, 1))
)
welch_coherence_mat = np.absolute(welch_coherence_mat) ** 2

Timing test for approximating eigenvalues with verious methods using real data.

In [ ]:
RandA = norm_win_spectra[300, :, :]
nreps = 10

t0 = time.time()
for i in range(nreps):
    Q, R = np.linalg.qr(RandA)
    # qr_approx2 = np.sum(np.absolute(R@R.transpose()), axis=0)**2
    qr_approx = np.diag(np.absolute(R @ R.transpose())) ** 2

    Q2, R2 = np.linalg.qr(RandA.T)
    qr_approx2 = np.diag(np.absolute(R2 @ R2.transpose())) ** 2
t1 = time.time()
qr_time = t1 - t0

print("QR time: ", qr_time)

t0 = time.time()
for i in range(nreps):
    coherence_mat = np.absolute(RandA @ np.conjugate(RandA.transpose())) ** 2
    eigenvals, _ = np.linalg.eig(coherence_mat)
t1 = time.time()
eig_time = t1 - t0

print("Eigenvalue time: ", eig_time)

In [ ]:
fsize = 12
qr_approx = np.sort(qr_approx)[::-1]
qr_approx2 = np.sort(qr_approx2)[::-1]
# qr_approx = np.sort(np.sum(R@R.transpose(), axis=0))[::-1]
qr_approx = qr_approx / np.sum(np.absolute(qr_approx))
qr_approx2 = qr_approx2 / np.sum(np.absolute(qr_approx2))

actual_eigenval = np.sort(eigenvals)[::-1]
actual_eigenval = actual_eigenval / np.sum(actual_eigenval)

plt.plot(
    actual_eigenval[: 6 * len(qr_approx)], "g-o", label="Actual eigen decomp"
)
plt.plot(qr_approx, "b-s", label="QR approx")
plt.plot(qr_approx2, "k-s", label="QR approx2")

plt.xlabel("Decending Order", fontsize=fsize)
plt.ylabel("Normalised Eignenvalue", fontsize=fsize)
plt.title("Eigenvalue Decay", fontsize=fsize)
# plt.yscale('log')
plt.legend(fontsize=fsize)

In [ ]:
RandA = norm_win_spectra[350, :, :]
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
Q1, R1 = np.linalg.qr(RandA.T)
# plt.matshow(np.abs(R1@R1.T))
plt.imshow(np.abs(R1 @ R1.T))
plt.colorbar()
plt.subplot(1, 2, 2)
Q1, R1 = np.linalg.qr(RandA)
plt.imshow(np.abs(R1 @ R1.T))
plt.colorbar()
# plt.subplot(1,3,3)

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(np.abs(R1))
plt.colorbar()
plt.title("R (event not at beginning of time series)", fontsize=15)
plt.subplot(1, 2, 2)
# plt.plot(frequencies)
qr_ = np.diag(np.absolute(R1 @ R1.transpose()))
plt.plot(qr_, "-o")
plt.title("QR Approximation of Eigenvalues", fontsize=15)
print(np.max(qr_) / np.sum(np.absolute(qr_)))

In [ ]:
# random matrix with same size as norm_win_spectra
random_A = np.random.rand(*norm_win_spectra.shape)
random_A = (
    random_A + 1j * np.random.rand(*norm_win_spectra.shape) + norm_win_spectra
)
RandAa = random_A[350, :, :]
welch_coherence_mat_rand = (
    np.absolute(RandAa @ np.conjugate(RandA.transpose())) ** 2
)

eigenvals, _ = np.linalg.eig(welch_coherence_mat_rand)
eigenvals = np.sort(eigenvals)[::-1]
eigenvals = eigenvals / np.sum(eigenvals)

welch_coherence_mat_rand = (
    np.absolute(RandAa @ np.conjugate(RandA.transpose())) ** 2
)

rand_eigenvals, _ = np.linalg.eig(welch_coherence_mat_rand)
rand_eigenvals = np.sort(rand_eigenvals)[::-1]
rand_eigenvals = rand_eigenvals / np.sum(rand_eigenvals)


Q, R = np.linalg.qr(RandA)
# qr_approx2 = np.sum(np.absolute(R@R.transpose()), axis=0)**2
qr_approx = np.diag(np.absolute(R @ R.transpose())) ** 2
qr_approx = qr_approx / np.sum(np.absolute(qr_approx))

Q, R = np.linalg.qr(RandAa)
# qr_approx2 = np.sum(np.absolute(R@R.transpose()), axis=0)**2
qr_approx_rand = np.diag(np.absolute(R @ R.transpose())) ** 2
qr_approx_rand = qr_approx_rand / np.sum(np.absolute(qr_approx_rand))

# rand_qr_approx = np.diag(np.absolute(
plt.plot(eigenvals, "r-o", label="Actual eigen decomp")
# plt.plot(rand_eigenvals, "g-o", label="rand plus Actual eigen decomp")
plt.plot(qr_approx, "b-s", label="QR approx")
plt.plot(qr_approx_rand, "k-s", label="rand plus QR approx")
plt.xlabel("Decending Order", fontsize=fsize)
plt.ylabel("Normalised Eignenvalue", fontsize=fsize)
plt.title("Eigenvalue Decay", fontsize=fsize)
plt.legend(fontsize=fsize)

In [ ]:
fsize = 15
# num_frames = coherence2.shape[0]
# data_2use = welch_coherence_mat.real
# data_2use = np.absolute(welch_coherence_mat)**2
# num_frames = int(data_2use.shape[0]/2)

num_frames = int(norm_win_spectra.shape[0] / 2)

eig_ratios2 = np.empty(num_frames)
eig_ratios_qr = np.empty(num_frames)
eig_ratios_qr2 = np.empty(num_frames)
eig_ratios_qr_diag = np.empty(num_frames)
eig_ratios_qr2_diag = np.empty(num_frames)
eig_ratios_qr_rand = np.empty(num_frames)
eig_ratios_svd = np.empty(num_frames)
eig_ratios_rsvd = np.empty(num_frames)
for d in range(num_frames):
    RandA = norm_win_spectra[d * 2]
    coherence_mat = np.absolute(RandA @ np.conjugate(RandA.transpose())) ** 2
    eigenvals, _ = np.linalg.eig(coherence_mat)
    # eigenvals, _ = np.linalg.eig(data_2use[d*2])
    eigenvals = np.sort(eigenvals)[::-1]
    eig_ratios2[d] = eigenvals[0] / np.sum(eigenvals)

    Q, R = np.linalg.qr(norm_win_spectra[d * 2])
    qr_approx = np.sort(np.sum(np.absolute(R @ R.transpose()), axis=0))[::-1]
    qr_approx_diag = np.sort(np.diag(np.absolute(R @ R.transpose())))[::-1]
    # qr_approx = np.sort(np.sum(R@R.transpose(), axis=0))[::-1]
    eig_ratios_qr[d] = qr_approx[0] / np.sum(np.absolute(qr_approx))
    eig_ratios_qr_diag[d] = qr_approx_diag[0] / np.sum(
        np.absolute(qr_approx_diag)
    )

    Q, R = np.linalg.qr(norm_win_spectra[d * 2].T)
    qr_approx2_diag = np.sort(np.diag(np.absolute(R @ R.transpose())))[::-1]
    # qr_approx2 = np.sort(np.sum(np.absolute(R@R.transpose()), axis=0))[::-1]
    qr_approx2, _ = np.linalg.eig(R @ (R.transpose().conj()))
    # qr_approx = np.sort(np.sum(R@R.transpose(), axis=0))[::-1]
    eig_ratios_qr2[d] = qr_approx2[0] / np.sum(np.absolute(qr_approx2))
    eig_ratios_qr2_diag[d] = qr_approx2_diag[0] / np.sum(
        np.absolute(qr_approx2_diag)
    )

    Q, R = np.linalg.qr(random_A[d * 2])
    # qr_approx2 = np.sum(np.absolute(R@R.transpose()), axis=0)**2
    qr_approx_rand = np.diag(np.absolute(R @ R.transpose())) ** 2
    eig_ratios_qr_rand[d] = qr_approx_rand[0] / np.sum(
        np.absolute(qr_approx_rand)
    )

    # U, S, Vh = np.linalg.svd(norm_win_spectra[d*2])
    # svd_approx = S**2
    # eig_ratios_svd[d] = svd_approx[0]/np.sum(svd_approx)

    # rU, rS, rVh = randomized_svd(norm_win_spectra[d*2], approx_rank)
    # rsvd_approx = rS**2
    # eig_ratios_rsvd[d] = rsvd_approx[0]/np.sum(rsvd_approx)

In [ ]:
plt.plot(eig_ratios2[:800], "r-", label="Actual eigen decomp")
# plt.plot(eig_ratios_qr[:800], "b-", label="QR approx")
plt.plot(eig_ratios_qr2[:800], "k-", label="QR approx2")
# plt.plot(eig_ratios_qr_diag[:800], "g-", label="QR approx diag")
# plt.plot(eig_ratios_qr2_diag[:800], "y-", label="QR approx2 diag")
# plt.plot(eig_ratios_qr_rand[:800], "m-", label="QR approx rand")

plt.xlabel("Descending Order", fontsize=fsize)
plt.ylabel("Detection Significance", fontsize=fsize)
plt.title("Detections", fontsize=fsize)
plt.legend(fontsize=fsize)